# Notebook 4 — Data Quality Report

**Purpose:** Measure and report data quality across all pipeline layers —
Bronze, Silver, and Gold.

In [0]:
# ============================================================
# We introduce a new table here: quality_report
# This is a metadata table — it stores metrics ABOUT the
# pipeline, not the pipeline data itself.
#
# In production this table would be queried by a monitoring
# dashboard (Grafana, Databricks SQL Dashboard) to show
# pipeline health over time. Every pipeline run appends
# a new set of rows so you can track quality trends.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DoubleType, TimestampType
)
from datetime import datetime

CATALOG_NAME = "iot_pipeline"
SCHEMA_NAME  = "sensors"

BRONZE_TABLE   = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_sensors"
SILVER_TABLE   = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_sensors"
GOLD_HOURLY    = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_hourly_stats"
GOLD_DAILY     = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_daily_health"
GOLD_ANOMALIES = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_anomalies"
QUALITY_TABLE  = f"{CATALOG_NAME}.{SCHEMA_NAME}.quality_report"

# Unique ID for this pipeline run — in production this would
# come from your orchestrator (Airflow, Databricks Workflows)
RUN_ID = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

print(f"Run ID        : {RUN_ID}")
print(f"Quality table : {QUALITY_TABLE}")

In [0]:

quality_schema = StructType([
    StructField("run_id",       StringType(),    nullable=False),
    StructField("layer",        StringType(),    nullable=False),  # bronze/silver/gold
    StructField("table_name",   StringType(),    nullable=False),
    StructField("check_name",   StringType(),    nullable=False),  # what was checked
    StructField("metric_value", DoubleType(),    nullable=True),   # the measured value
    StructField("threshold",    DoubleType(),    nullable=True),   # acceptable limit
    StructField("status",       StringType(),    nullable=False),  # PASS / WARN / FAIL
    StructField("checked_at",   TimestampType(), nullable=False),
])

# Helper function to build a quality metric row
def make_metric(layer, table, check_name, value, threshold, status):
    """
    Returns a dict representing one quality check result.
    We use a function so every row has the same structure —
    no risk of accidentally missing a field.
    """
    return {
        "run_id":       RUN_ID,
        "layer":        layer,
        "table_name":   table,
        "check_name":   check_name,
        "metric_value": float(value) if value is not None else None,
        "threshold":    float(threshold) if threshold is not None else None,
        "status":       status,
        "checked_at":   datetime.utcnow(),
    }

print("Schema and helper function defined.")

In [0]:
# ============================================================
# We check the Bronze layer for the three problems we
# intentionally injected in Notebook 1.
# Each check produces one row in our quality report.
#
# THRESHOLDS explained:
# - null_rate_pct: we accept up to 5% nulls in Bronze
#   (it's raw data, some nulls are expected)
# - duplicate_rate_pct: we accept up to 3% duplicates in Bronze
# - out_of_range_pct: we accept up to 10% in Bronze
#   (range violations get cleaned in Silver, not Bronze)
#
# Bronze thresholds are deliberately lenient — Bronze is raw.
# Silver thresholds (next cell) are strict — Silver is clean.
# ============================================================

print("Running Bronze checks...")

df_bronze = spark.table(BRONZE_TABLE)
bronze_total = df_bronze.count()

# Check 1 — null rate
bronze_nulls = df_bronze.filter(F.col("value").isNull()).count()
bronze_null_pct = round(bronze_nulls / bronze_total * 100, 2)
null_status = "PASS" if bronze_null_pct <= 5.0 else "FAIL"

# Check 2 — duplicate rate
bronze_dupes = bronze_total - df_bronze.dropDuplicates(["event_id"]).count()
bronze_dupe_pct = round(bronze_dupes / bronze_total * 100, 2)
dupe_status = "PASS" if bronze_dupe_pct <= 3.0 else "FAIL"

# Check 3 — row count sanity (expect at least 9,000 rows)
count_status = "PASS" if bronze_total >= 9000 else "FAIL"

bronze_metrics = [
    make_metric("bronze", BRONZE_TABLE, "row_count",         bronze_total,    9000,  count_status),
    make_metric("bronze", BRONZE_TABLE, "null_rate_pct",     bronze_null_pct, 5.0,   null_status),
    make_metric("bronze", BRONZE_TABLE, "duplicate_rate_pct",bronze_dupe_pct, 3.0,   dupe_status),
]

print(f"  Row count       : {bronze_total:,}       [{count_status}]")
print(f"  Null rate       : {bronze_null_pct}%  [{null_status}]")
print(f"  Duplicate rate  : {bronze_dupe_pct}%  [{dupe_status}]")

In [0]:
# ============================================================
# Silver thresholds are STRICT because Silver is supposed
# to be clean. Zero nulls, zero duplicates, zero out-of-range.
# Any violation here means the Silver transformation in
# Notebook 2 has a bug.
#
# We also check that row count didn't drop TOO much from
# Bronze. We expect to lose ~10% of rows during cleaning.
# If we lost 50%, something went wrong.
# ============================================================

print("Running Silver checks...")

df_silver = spark.table(SILVER_TABLE)
silver_total = df_silver.count()

# Check 1 — zero nulls expected
silver_nulls = df_silver.filter(F.col("value").isNull()).count()
silver_null_status = "PASS" if silver_nulls == 0 else "FAIL"

# Check 2 — zero duplicates expected
silver_dupes = silver_total - df_silver.dropDuplicates(["event_id"]).count()
silver_dupe_status = "PASS" if silver_dupes == 0 else "FAIL"

# Check 3 — out-of-range values should be zero
range_violations = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {SILVER_TABLE}
    WHERE
        (sensor_type = 'temperature' AND (value < 60  OR value > 120)) OR
        (sensor_type = 'pressure'    AND (value < 1.0 OR value > 10))  OR
        (sensor_type = 'vibration'   AND (value < 0.01 OR value > 2.0))
""").collect()[0]["n"]
range_status = "PASS" if range_violations == 0 else "FAIL"

# Check 4 — data loss check (Silver should be 80-98% of Bronze)
retention_pct = round(silver_total / bronze_total * 100, 2)
# WARN if we kept less than 85% — unexpected data loss
# FAIL if we kept less than 70% — something is seriously wrong
if retention_pct >= 85:
    retention_status = "PASS"
elif retention_pct >= 70:
    retention_status = "WARN"
else:
    retention_status = "FAIL"

silver_metrics = [
    make_metric("silver", SILVER_TABLE, "row_count",          silver_total,    None,  "PASS"),
    make_metric("silver", SILVER_TABLE, "null_count",         silver_nulls,    0,     silver_null_status),
    make_metric("silver", SILVER_TABLE, "duplicate_count",    silver_dupes,    0,     silver_dupe_status),
    make_metric("silver", SILVER_TABLE, "range_violations",   range_violations,0,     range_status),
    make_metric("silver", SILVER_TABLE, "retention_pct",      retention_pct,   85.0,  retention_status),
]

print(f"  Row count         : {silver_total:,}      [PASS]")
print(f"  Null count        : {silver_nulls}          [{silver_null_status}]")
print(f"  Duplicate count   : {silver_dupes}          [{silver_dupe_status}]")
print(f"  Range violations  : {range_violations}          [{range_status}]")
print(f"  Retention from Bronze: {retention_pct}%  [{retention_status}]")

In [0]:
# ============================================================
# Gold checks focus on COMPLETENESS and CONSISTENCY:
# - Do we have data for all expected machines?
# - Do we have data for all expected days?
# - Does the anomaly rate look reasonable?
#
# We don't re-check nulls and duplicates here — Silver already
# guaranteed clean data. Gold checks are about whether the
# aggregation produced sensible results.
#
# This is an important distinction: each layer has different
# quality concerns. Bronze = raw completeness. Silver = cleanliness.
# Gold = business logic correctness.
# ============================================================

print("Running Gold checks...")

# Check 1 — all 5 machines present in daily health table
machine_count = spark.sql(f"""
    SELECT COUNT(DISTINCT machine_id) AS n
    FROM {GOLD_DAILY}
""").collect()[0]["n"]
machine_status = "PASS" if machine_count == 5 else "FAIL"

# Check 2 — health scores are in valid range (0-100)
invalid_scores = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {GOLD_DAILY}
    WHERE health_score_pct < 0 OR health_score_pct > 100
""").collect()[0]["n"]
score_status = "PASS" if invalid_scores == 0 else "FAIL"

# Check 3 — anomaly rate is plausible (expect 3-15% of Silver rows)
anomaly_total = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD_ANOMALIES}").collect()[0]["n"]
anomaly_rate  = round(anomaly_total / silver_total * 100, 2)
# Too low (< 1%) might mean the detection isn't working
# Too high (> 20%) might mean the threshold is wrong
if 1.0 <= anomaly_rate <= 20.0:
    anomaly_status = "PASS"
elif anomaly_rate < 1.0:
    anomaly_status = "WARN"
else:
    anomaly_status = "FAIL"

# Check 4 — hourly stats cover all 7 days of data
day_count = spark.sql(f"""
    SELECT COUNT(DISTINCT date) AS n
    FROM {GOLD_HOURLY}
""").collect()[0]["n"]
day_status = "PASS" if day_count == 7 else "WARN"

gold_metrics = [
    make_metric("gold", GOLD_DAILY,    "distinct_machines",  machine_count,  5,    machine_status),
    make_metric("gold", GOLD_DAILY,    "invalid_scores",     invalid_scores, 0,    score_status),
    make_metric("gold", GOLD_ANOMALIES,"anomaly_rate_pct",   anomaly_rate,   20.0, anomaly_status),
    make_metric("gold", GOLD_HOURLY,   "days_covered",       day_count,      7,    day_status),
]

print(f"  Distinct machines   : {machine_count}  [{machine_status}]")
print(f"  Invalid health scores: {invalid_scores}  [{score_status}]")
print(f"  Anomaly rate        : {anomaly_rate}%  [{anomaly_status}]")
print(f"  Days covered        : {day_count}  [{day_status}]")

## Cross-layer lineage check

This is the most important check in the entire pipeline.

It verifies that data flows correctly from Bronze → Silver → Gold
and that we can account for every row that was removed and why.

In [0]:

hourly_total  = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD_HOURLY}").collect()[0]["n"]
daily_total   = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD_DAILY}").collect()[0]["n"]
removed_total = bronze_total - silver_total

print("=" * 55)
print("  CROSS-LAYER LINEAGE REPORT")
print("=" * 55)
print(f"  Bronze rows (raw)          : {bronze_total:>7,}")
print(f"  Rows removed in cleaning   : {removed_total:>7,}  ({round(removed_total/bronze_total*100,1)}%)")
print(f"  Silver rows (clean)        : {silver_total:>7,}  ({round(silver_total/bronze_total*100,1)}% of Bronze)")
print(f"  Gold hourly stats rows     : {hourly_total:>7,}  (aggregated)")
print(f"  Gold daily health rows     : {daily_total:>7,}  (aggregated)")
print(f"  Gold anomaly rows          : {anomaly_total:>7,}  ({anomaly_rate}% of Silver)")
print("=" * 55)

# Log lineage metrics too
lineage_metrics = [
    make_metric("lineage", "cross_layer", "bronze_to_silver_retention_pct", retention_pct,  85.0, retention_status),
    make_metric("lineage", "cross_layer", "rows_removed_cleaning",          removed_total,  None, "PASS"),
    make_metric("lineage", "cross_layer", "anomaly_rate_pct",               anomaly_rate,   20.0, anomaly_status),
]

In [0]:
# ============================================================
# We combine all metrics into one DataFrame and write to
# the quality_report Delta table using APPEND mode.
#
# WHY APPEND and not OVERWRITE:
# This is critical. If we used overwrite, each pipeline run
# would delete all previous quality history. With append,
# every run adds new rows tagged with run_id and checked_at.
# Over time this builds a complete history of pipeline quality.
# ============================================================

all_metrics = bronze_metrics + silver_metrics + gold_metrics + lineage_metrics

df_quality = spark.createDataFrame(all_metrics, schema=quality_schema)

# Create table on first run, append on subsequent runs
(
    df_quality
    .write
    .format("delta")
    .mode("append")           # APPEND — never overwrite quality history
    .saveAsTable(QUALITY_TABLE)
)

print(f"Quality report written: {len(all_metrics)} metrics for run {RUN_ID}")

In [0]:
# ============================================================
# Now we query our own quality report table to display
# a clean summary. This is what a monitoring dashboard
# would show after every pipeline run.
# ============================================================

print(f"=== Quality report for run: {RUN_ID} ===\n")

# Overall pass/fail summary
spark.sql(f"""
    SELECT
        layer,
        COUNT(*)                                    AS total_checks,
        SUM(CASE WHEN status = 'PASS' THEN 1 END)  AS passed,
        SUM(CASE WHEN status = 'WARN' THEN 1 END)  AS warned,
        SUM(CASE WHEN status = 'FAIL' THEN 1 END)  AS failed
    FROM {QUALITY_TABLE}
    WHERE run_id = '{RUN_ID}'
    GROUP BY layer
    ORDER BY layer
""").show()

# Full detail — all checks
print("=== All checks detail ===")
display(
    spark.sql(f"""
        SELECT
            layer,
            check_name,
            ROUND(metric_value, 2)  AS metric_value,
            threshold,
            status
        FROM {QUALITY_TABLE}
        WHERE run_id = '{RUN_ID}'
        ORDER BY layer, check_name
    """)
)

# Any failures or warnings
failures = spark.sql(f"""
    SELECT * FROM {QUALITY_TABLE}
    WHERE run_id = '{RUN_ID}'
    AND status IN ('FAIL', 'WARN')
""").count()

if failures == 0:
    print("\n  All checks passed for this run.")
else:
    print(f"\n  {failures} check(s) need attention:")
    spark.sql(f"""
        SELECT layer, check_name, metric_value, threshold, status
        FROM {QUALITY_TABLE}
        WHERE run_id = '{RUN_ID}'
        AND status IN ('FAIL', 'WARN')
    """).show()

In [0]:
# ============================================================
# This query shows the POWER of writing metrics to a table.
# If you run the full pipeline multiple times, this query
# shows you how quality metrics change over time.
#
# In a real pipeline with daily runs, this would show you
# if null rates are slowly increasing (sensor degradation)
# or if row counts are dropping (upstream data source issue).
# ============================================================

print("=== Quality trend across all runs ===")
spark.sql(f"""
    SELECT
        run_id,
        checked_at,
        SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) AS passed,
        SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) AS warned,
        SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS failed,
        COUNT(*) AS total_checks
    FROM {QUALITY_TABLE}
    GROUP BY run_id, checked_at
    ORDER BY checked_at DESC
""").show()